# Importaciones de librerías

In [63]:
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, make_scorer, confusion_matrix, average_precision_score
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import GridSearchCV
import joblib

# Carga de datos fichero 01

In [64]:
df = pd.read_csv("../../data/hospital_deterioration_ml_ready_prepared.csv")
X = df.drop(columns=['deterioration_next_12h'])
y = df['deterioration_next_12h']

with open("../config/preprocesamiento_api.json", "r") as f:
    config = json.load(f)

variables_a_escalar = config["variables_a_escalar"]
peso_desbalanceo = config["peso_desbalanceo"]

## Configuración grupos de entrenamiento

In [65]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=33)
scaler = StandardScaler()

# Evaluación de modelos base
## Clasificador aleatorio

In [66]:
from sklearn.dummy import DummyClassifier

dummy_auc = []
dummy_auc_pr = []
dummy_recall = []
dummy_specificity = []
dummy_precision = []
dummy_f1 = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model_dummy = DummyClassifier(strategy='most_frequent', random_state=777)

    model_dummy.fit(X_train, y_train)
    
    preds = model_dummy.predict(X_val)

    probs = model_dummy.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    try:
        auc_pr = average_precision_score(y_val, probs)
    except ValueError:
        auc_pr = 0.0
    
    dummy_auc.append(roc_auc_score(y_val, probs))
    dummy_auc_pr.append(auc_pr)
    dummy_recall.append(recall_score(y_val, preds, zero_division=0))
    dummy_specificity.append(spec)
    dummy_precision.append(precision_score(y_val, preds, zero_division=0))
    dummy_f1.append(f1_score(y_val, preds, zero_division=0))

print("--- RESULTADOS BASELINE: DUMMY CLASSIFIER ---")
print(f"AUC-ROC promedio:  {np.mean(dummy_auc):.4f}")
print(f"AUC-PR promedio:  {np.mean(dummy_auc_pr):.4f} ± {np.std(dummy_auc_pr):.4f}")
print(f"Recall promedio:   {np.mean(dummy_recall):.4f}")
print(f"Especificidad promedio:  {np.mean(dummy_specificity):.4f} ± {np.std(dummy_specificity):.4f}")
print(f"Precisión promedia: {np.mean(dummy_precision):.4f}")
print(f"F1-Score promedio:  {np.mean(dummy_f1):.4f} ± {np.std(dummy_f1):.4f}")

--- RESULTADOS BASELINE: DUMMY CLASSIFIER ---
AUC-ROC promedio:  0.5000
AUC-PR promedio:  0.0541 ± 0.0000
Recall promedio:   0.0000
Especificidad promedio:  1.0000 ± 0.0000
Precisión promedia: 0.0000
F1-Score promedio:  0.0000 ± 0.0000


## Regresión logística

In [67]:
from sklearn.linear_model import LogisticRegression

lr_auc = []
lr_auc_pr = []
lr_recall = []
lr_specificity = []
lr_precision = []
lr_f1 = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    X_train[variables_a_escalar] = scaler.fit_transform(X_train[variables_a_escalar])
    X_val[variables_a_escalar] = scaler.transform(X_val[variables_a_escalar])
    
    model_lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=369)
    model_lr.fit(X_train, y_train)
    
    preds = model_lr.predict(X_val)
    probs = model_lr.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    lr_auc.append(roc_auc_score(y_val, probs))
    lr_auc_pr.append(average_precision_score(y_val, probs))
    lr_recall.append(recall_score(y_val, preds))
    lr_specificity.append(spec)
    lr_precision.append(precision_score(y_val, preds))
    lr_f1.append(f1_score(y_val, preds, zero_division=0))

print("--- RESULTADOS BASELINE: REGRESIÓN LOGÍSTICA ---")
print(f"AUC-ROC promedio:  {np.mean(lr_auc):.4f}")
print(f"AUC-PR promedio:  {np.mean(lr_auc_pr):.4f} ± {np.std(lr_auc_pr):.4f}")
print(f"Recall promedio:   {np.mean(lr_recall):.4f}")
print(f"Especificidad promedio:  {np.mean(lr_specificity):.4f} ± {np.std(lr_specificity):.4f}")
print(f"Precisión promedia: {np.mean(lr_precision):.4f}")
print(f"F1-Score promedio:  {np.mean(lr_f1):.4f} ± {np.std(lr_f1):.4f}")

--- RESULTADOS BASELINE: REGRESIÓN LOGÍSTICA ---
AUC-ROC promedio:  0.8161
AUC-PR promedio:  0.2558 ± 0.0039
Recall promedio:   0.7235
Especificidad promedio:  0.7555 ± 0.0012
Precisión promedia: 0.1447
F1-Score promedio:  0.2411 ± 0.0021


## Árbol simple

In [68]:
historico_auc = []
historico_auc_pr = []
historico_recall = []
historico_specificity = []
historico_precision = []
historico_f1 = []

scaler = StandardScaler()

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model_baseline = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=42)
    
    model_baseline.fit(X_train, y_train)
    
    preds = model_baseline.predict(X_val)
    probs = model_baseline.predict_proba(X_val)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    historico_auc.append(roc_auc_score(y_val, probs))
    historico_auc_pr.append(average_precision_score(y_val, probs))
    historico_recall.append(recall_score(y_val, preds, zero_division=0))
    historico_specificity.append(spec)
    historico_precision.append(precision_score(y_val, preds, zero_division=0))
    historico_f1.append(f1_score(y_val, preds, zero_division=0))

print("--- RESULTADOS BASELINE (ÁRBOL DE DECISIÓN SIMPLE) ---")
print(f"AUC-ROC promedio:       {np.mean(historico_auc):.4f} ± {np.std(historico_auc):.4f}")
print(f"AUC-PR promedio:        {np.mean(historico_auc_pr):.4f} ± {np.std(historico_auc_pr):.4f}")
print(f"Recall (Sensibilidad): {np.mean(historico_recall):.4f} ± {np.std(historico_recall):.4f}")
print(f"Especificidad promedio: {np.mean(historico_specificity):.4f} ± {np.std(historico_specificity):.4f}")
print(f"Precisión promedia:     {np.mean(historico_precision):.4f} ± {np.std(historico_precision):.4f}")
print(f"F1-Score promedio:      {np.mean(historico_f1):.4f} ± {np.std(historico_f1):.4f}")

--- RESULTADOS BASELINE (ÁRBOL DE DECISIÓN SIMPLE) ---
AUC-ROC promedio:       0.8672 ± 0.0047
AUC-PR promedio:        0.2619 ± 0.0076
Recall (Sensibilidad): 0.7848 ± 0.0399
Especificidad promedio: 0.7998 ± 0.0353
Precisión promedia:     0.1856 ± 0.0172
F1-Score promedio:      0.2994 ± 0.0206


# Comparativa

In [69]:
tabla_baselines = pd.DataFrame({
    'Modelo Baseline': ['Clasificador Dummy', 'Regresión Logística', 'Árbol de Decisión Simple'],
    'AUC-ROC': [np.mean(dummy_auc), np.mean(lr_auc), np.mean(historico_auc)],
    'AUC-PR': [np.mean(dummy_auc_pr), np.mean(lr_auc_pr), np.mean(historico_auc_pr)],
    'Recall (Sensibilidad)': [np.mean(dummy_recall), np.mean(lr_recall), np.mean(historico_recall)],
    'Especificidad': [np.mean(dummy_specificity), np.mean(lr_specificity), np.mean(historico_specificity)],
    'Precisión': [np.mean(dummy_precision), np.mean(lr_precision), np.mean(historico_precision)],
    'F1-Score': [np.mean(dummy_f1), np.mean(lr_f1), np.mean(historico_f1)]
})

display(tabla_baselines.round(4))

,Modelo Baseline,AUC-ROC,AUC-PR,Recall (Sensibilidad),Especificidad,Precisión,F1-Score
0,Clasificador Dummy,0.5000,0.0541,0.0000,1.0000,0.0000,0.0000
1,Regresión Logística,0.8161,0.2558,0.7235,0.7555,0.1447,0.2411
2,Árbol de Decisión Simple,0.8672,0.2619,0.7848,0.7998,0.1856,0.2994


# Configuración del entrenamiento

In [70]:
especificidad_scorer = make_scorer(recall_score, pos_label=0)

scorers = {
    'Recall': 'recall',
    'Especificidad': especificidad_scorer,
    'Precisión': 'precision',
    'F1-Score': 'f1',
    'AUC-ROC': 'roc_auc',
    'AUC-PR': 'average_precision'
}

In [71]:
modelo_xgb = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=config["peso_desbalanceo"],
    eval_metric="auc",
    random_state=30,
    tree_method="hist"
)

In [72]:
modelo_lgb = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    scale_pos_weight=config["peso_desbalanceo"],
    metric="auc",
    random_state=86,
    verbosity=-1
)

## Entrenamiento xgboost

In [73]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    modelo_xgb.fit(X_tr, y_tr)
    
    preds_proba_xgb_0 = modelo_xgb.predict_proba(X_val)[:, 1]
    fold_auc_xgb_0 = roc_auc_score(y_val, preds_proba_xgb_0)
    fold_recall_xgb_0 = recall_score(y_val, modelo_xgb.predict(X_val))

## Entrenamiento LightGBM

In [74]:
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    # Entrenamiento limpio
    modelo_lgb.fit(X_tr, y_tr)
    
    # Evaluamos el fold actual
    preds_proba_lgb_0 = modelo_lgb.predict_proba(X_val)[:, 1]
    fold_auc_lgb_0 = roc_auc_score(y_val, preds_proba_lgb_0)
    fold_recall_lgb_0 = recall_score(y_val, modelo_lgb.predict(X_val))

## Modelos con distintos hiperparámetros

In [75]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [4, 5, 6, 7, 8],
    'subsample': [0.8, 0.9, 1.0]
}

### XGBoost

In [76]:
xgb_base = xgb.XGBClassifier(
    scale_pos_weight=config["peso_desbalanceo"],
    eval_metric="auc",
    random_state=30,
    tree_method="hist"
)

grid_search_xgb = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    scoring=scorers,
    refit='Recall',
    cv=skf,
    verbose=1,
    n_jobs=-1
)

grid_search_xgb.fit(X, y)

resultados_xgb = grid_search_xgb.cv_results_

df_resultados_xgb = pd.DataFrame({
    'n_estimators': resultados_xgb['param_n_estimators'],
    'learning_rate': resultados_xgb['param_learning_rate'],
    'max_depth': resultados_xgb['param_max_depth'],
    'subsample': resultados_xgb['param_subsample'],
    'Recall Medio': resultados_xgb['mean_test_Recall'],
    'Especificidad Media': resultados_xgb['mean_test_Especificidad'],
    'AUC-ROC Medio': resultados_xgb['mean_test_AUC-ROC'],
    'AUC-PR Medio': resultados_xgb['mean_test_AUC-PR'],
    'F1-Score Medio': resultados_xgb['mean_test_F1-Score'],
    'Precisión Media': resultados_xgb['mean_test_Precisión']
})

display(df_resultados_xgb.sort_values(by='Recall Medio', ascending=False))

modelo_final_xgb = grid_search_xgb.best_estimator_

Fitting 5 folds for each of 135 candidates, totalling 675 fits


,n_estimators,learning_rate,max_depth,subsample,Recall Medio,Especificidad Media,AUC-ROC Medio,AUC-PR Medio,F1-Score Medio,Precisión Media
132,200,0.10,8,0.8,0.894861,0.960741,0.983142,0.829828,0.693231,0.565790
133,200,0.10,8,0.9,0.892780,0.959221,0.982743,0.826892,0.685125,0.555885
134,200,0.10,8,1.0,0.891894,0.955826,0.981349,0.819548,0.669396,0.535776
123,200,0.10,7,0.8,0.890611,0.940556,0.976372,0.780559,0.607776,0.461304
124,200,0.10,7,0.9,0.890257,0.938873,0.975730,0.776336,0.601565,0.454273
...,...,...,...,...,...,...,...,...,...,...
4,100,0.01,4,0.9,0.787198,0.831756,0.894247,0.434911,0.332774,0.210984
5,100,0.01,4,1.0,0.784896,0.832442,0.894107,0.429944,0.332797,0.211166
0,50,0.01,4,0.8,0.770553,0.829762,0.884256,0.354267,0.324495,0.205525
1,50,0.01,4,0.9,0.767719,0.830623,0.883790,0.348346,0.324561,0.205787


### LightGBM

In [77]:
lgb_base = lgb.LGBMClassifier(
    scale_pos_weight=config["peso_desbalanceo"],
    eval_metric="auc",
    random_state=88,
    tree_method="hist"
)

grid_search_lgb = GridSearchCV(
    estimator=lgb_base,
    param_grid=param_grid,
    scoring=scorers,
    refit='Recall',
    cv=skf,
    verbose=1,
    n_jobs=-1
)

grid_search_lgb.fit(X, y)

resultados_lgb = grid_search_lgb.cv_results_

df_resultados_lgb = pd.DataFrame({
    'n_estimators': resultados_lgb['param_n_estimators'],
    'learning_rate': resultados_lgb['param_learning_rate'],
    'max_depth': resultados_lgb ['param_max_depth'],
    'subsample': resultados_lgb['param_subsample'],
    'Recall Medio': resultados_lgb['mean_test_Recall'],
    'Especificidad Media': resultados_lgb['mean_test_Especificidad'],
    'AUC-ROC Medio': resultados_lgb['mean_test_AUC-ROC'],
    'AUC-PR Medio': resultados_lgb['mean_test_AUC-PR'],
    'Precisión Media': resultados_lgb['mean_test_Precisión'],
    'F1-Score Medio': resultados_lgb['mean_test_F1-Score']
})

display(df_resultados_lgb.sort_values(by='Recall Medio', ascending=False))

modelo_final_lgb = grid_search_lgb.best_estimator_

Fitting 5 folds for each of 135 candidates, totalling 675 fits


,n_estimators,learning_rate,max_depth,subsample,Recall Medio,Especificidad Media,AUC-ROC Medio,AUC-PR Medio,Precisión Media,F1-Score Medio
133,200,0.10,8,0.9,0.883572,0.910073,0.965276,0.719036,0.359611,0.511167
132,200,0.10,8,0.8,0.883572,0.910073,0.965276,0.719036,0.359611,0.511167
134,200,0.10,8,1.0,0.883572,0.910073,0.965276,0.719036,0.359611,0.511167
124,200,0.10,7,0.9,0.880473,0.908806,0.964444,0.716340,0.355569,0.506565
125,200,0.10,7,1.0,0.880473,0.908806,0.964444,0.716340,0.355569,0.506565
...,...,...,...,...,...,...,...,...,...,...
38,50,0.01,8,1.0,0.000000,1.000000,0.917363,0.537279,0.000000,0.000000
10,50,0.01,5,0.9,0.000000,1.000000,0.907197,0.479677,0.000000,0.000000
9,50,0.01,5,0.8,0.000000,1.000000,0.907197,0.479677,0.000000,0.000000
18,50,0.01,6,0.8,0.000000,1.000000,0.913521,0.505941,0.000000,0.000000


## Optimización del umbral de decisión

### Calcular probabilidades de datos de test

In [78]:
y_probs = cross_val_predict(
    modelo_final_xgb,
    X, y,
    cv=skf,
    method='predict_proba',
    n_jobs=-1
)[:, 1]

### Evaluación del rendimiento con distintos umbrales

In [79]:
umbrales = np.linspace(0.05, 0.95, 91)
registro_umbrales = []

for u in umbrales:
    preds_bajo_umbral = (y_probs >= u).astype(int)
    
    rec = recall_score(y, preds_bajo_umbral, zero_division=0)
    prec = precision_score(y, preds_bajo_umbral, zero_division=0)
    f1 = f1_score(y, preds_bajo_umbral, zero_division=0)
    
    registro_umbrales.append({
        'Umbral': u,
        'Recall (Sensibilidad)': rec,
        'Precisión': prec,
        'F1-Score': f1
    })

df_umbrales = pd.DataFrame(registro_umbrales)

In [80]:
criterio_clinico = df_umbrales[df_umbrales['Recall (Sensibilidad)'] >= 0.89]

if not criterio_clinico.empty:
    fila_optima = criterio_clinico.sort_values(by='F1-Score', ascending=False).iloc[0]
else:
    fila_optima = df_umbrales.sort_values(by='F1-Score', ascending=False).iloc[0]

umbral_optimo = round(fila_optima['Umbral'], 4)
display(df_umbrales.sort_values(by='F1-Score', ascending=False).head(10).round(4))

,Umbral,Recall (Sensibilidad),Precisión,F1-Score
71,0.76,0.7642,0.7794,0.7717
70,0.75,0.7705,0.7718,0.7712
72,0.77,0.7554,0.7868,0.7708
69,0.74,0.7783,0.7633,0.7707
73,0.78,0.7478,0.7940,0.7702
68,0.73,0.7849,0.7556,0.7699
74,0.79,0.7394,0.8021,0.7694
67,0.72,0.7912,0.7483,0.7692
66,0.71,0.7982,0.7412,0.7686
75,0.80,0.7312,0.8091,0.7681


In [81]:
df_todos_baselines = pd.DataFrame([
    {
        'Familia': 'Baseline',
        'Algoritmo': 'Clasificador Dummy',
        'Configuracion / Hiperparametros': "strategy='most_frequent'",
        'Recall Medio': np.mean(dummy_recall),
        'Recall Std': np.std(dummy_recall),
        'Especificidad Media': np.mean(dummy_specificity),
        'Especificidad Std': np.std(dummy_specificity),
        'AUC-ROC Medio': np.mean(dummy_auc),
        'AUC-ROC Std': np.std(dummy_auc),
        'AUC-PR Medio': np.mean(dummy_auc_pr),
        'AUC-PR Std': np.std(dummy_auc_pr),
        'Precision Media': np.mean(dummy_precision),
        'Precision Std': np.std(dummy_precision),
        'F1-Score Medio': np.mean(dummy_f1),
        'F1-Score Std': np.std(dummy_f1)
    },
    {
        'Familia': 'Baseline',
        'Algoritmo': 'Regresión Logística',
        'Configuracion / Hiperparametros': "class_weight='balanced', max_iter=1000",
        'Recall Medio': np.mean(lr_recall),
        'Recall Std': np.std(lr_recall),
        'Especificidad Media': np.mean(lr_specificity),
        'Especificidad Std': np.std(lr_specificity),
        'AUC-ROC Medio': np.mean(lr_auc),
        'AUC-ROC Std': np.std(lr_auc),
        'AUC-PR Medio': np.mean(lr_auc_pr),
        'AUC-PR Std': np.std(lr_auc_pr),
        'Precision Media': np.mean(lr_precision),
        'Precision Std': np.std(lr_precision),
        'F1-Score Medio': np.mean(lr_f1),
        'F1-Score Std': np.std(lr_f1)
    },
    {
        'Familia': 'Baseline',
        'Algoritmo': 'Árbol de Decisión Simple',
        'Configuracion / Hiperparametros': "max_depth=5, class_weight='balanced'",
        'Recall Medio': np.mean(historico_recall),
        'Recall Std': np.std(historico_recall),
        'Especificidad Media': np.mean(historico_specificity),
        'Especificidad Std': np.std(historico_specificity),
        'AUC-ROC Medio': np.mean(historico_auc),
        'AUC-ROC Std': np.std(historico_auc),
        'AUC-PR Medio': np.mean(historico_auc_pr),
        'AUC-PR Std': np.std(historico_auc_pr),
        'Precision Media': np.mean(historico_precision),
        'Precision Std': np.std(historico_precision),
        'F1-Score Medio': np.mean(historico_f1),
        'F1-Score Std': np.std(historico_f1)
    }
])

res_xgb = grid_search_xgb.cv_results_
df_todos_xgb = pd.DataFrame({
    'Familia': 'XGBoost',
    'Algoritmo': 'XGBClassifier',
    'Configuracion / Hiperparametros': [str(p) for p in res_xgb['params']],
    'n_estimators': res_xgb['param_n_estimators'].data,
    'learning_rate': res_xgb['param_learning_rate'].data,
    'max_depth': res_xgb['param_max_depth'].data,
    'subsample': res_xgb['param_subsample'].data,
    'Recall Medio': res_xgb['mean_test_Recall'],
    'Recall Std': res_xgb['std_test_Recall'],
    'Especificidad Media': res_xgb['mean_test_Especificidad'],
    'Especificidad Std': res_xgb['std_test_Especificidad'],
    'AUC-ROC Medio': res_xgb['mean_test_AUC-ROC'],
    'AUC-ROC Std': res_xgb['std_test_AUC-ROC'],
    'AUC-PR Medio': res_xgb['mean_test_AUC-PR'],
    'AUC-PR Std': res_xgb['std_test_AUC-PR'],
    'Precision Media': res_xgb['mean_test_Precisión'],
    'Precision Std': res_xgb['std_test_Precisión'],
    'F1-Score Medio': res_xgb['mean_test_F1-Score'],
    'F1-Score Std': res_xgb['std_test_F1-Score']
}).sort_values(by='Recall Medio', ascending=False)

res_lgb = grid_search_lgb.cv_results_
df_todos_lgb = pd.DataFrame({
    'Familia': 'LightGBM',
    'Algoritmo': 'LGBMClassifier',
    'Configuracion / Hiperparametros': [str(p) for p in res_lgb['params']],
    'n_estimators': res_lgb['param_n_estimators'].data,
    'learning_rate': res_lgb['param_learning_rate'].data,
    'max_depth': res_lgb['param_max_depth'].data,
    'subsample': res_lgb['param_subsample'].data,
    'Recall Medio': res_lgb['mean_test_Recall'],
    'Recall Std': res_lgb['std_test_Recall'],
    'Especificidad Media': res_lgb['mean_test_Especificidad'],
    'Especificidad Std': res_lgb['std_test_Especificidad'],
    'AUC-ROC Medio': res_lgb['mean_test_AUC-ROC'],
    'AUC-ROC Std': res_lgb['std_test_AUC-ROC'],
    'AUC-PR Medio': res_lgb['mean_test_AUC-PR'],
    'AUC-PR Std': res_lgb['std_test_AUC-PR'],
    'Precision Media': res_lgb['mean_test_Precisión'],
    'Precision Std': res_lgb['std_test_Precisión'],
    'F1-Score Medio': res_lgb['mean_test_F1-Score'],
    'F1-Score Std': res_lgb['std_test_F1-Score']
}).sort_values(by='Recall Medio', ascending=False)

columnas_comunes = [
    'Familia', 'Algoritmo', 'Configuracion / Hiperparametros',
    'Recall Medio', 'Recall Std',
    'Especificidad Media', 'Especificidad Std',
    'AUC-ROC Medio', 'AUC-ROC Std',
    'AUC-PR Medio', 'AUC-PR Std',
    'Precision Media', 'Precision Std',
    'F1-Score Medio', 'F1-Score Std'
]

df_todos_unificados = pd.concat([
    df_todos_baselines[columnas_comunes],
    df_todos_xgb[columnas_comunes],
    df_todos_lgb[columnas_comunes]
], ignore_index=True)
df_todos_unificados.to_csv("01_todos_los_modelos_consolidados.csv", index=False, sep=";", encoding="utf-8-sig")
df_todos_xgb.to_csv("02_xgboost_135_modelos.csv", index=False, sep=";", encoding="utf-8-sig")
df_todos_lgb.to_csv("03_lightgbm_135_modelos.csv", index=False, sep=";", encoding="utf-8-sig")
df_todos_baselines.to_csv("04_baselines.csv", index=False, sep=";", encoding="utf-8-sig")
df_umbrales.to_csv("05_calibracion_umbrales_xgb.csv", index=False, sep=";", encoding="utf-8-sig")

print(f"✅ Exportados con éxito:")
print(f" - 'todos_los_modelos_evaluados.csv' con {len(df_todos_unificados)} modelos registrados.")
print(f" - 'reporte_completo_modelado.xlsx' con 5 hojas separadas (Consolidado, XGBoost, LightGBM, Baselines y Umbrales).")

✅ Exportados con éxito:
 - 'todos_los_modelos_evaluados.csv' con 273 modelos registrados.
 - 'reporte_completo_modelado.xlsx' con 5 hojas separadas (Consolidado, XGBoost, LightGBM, Baselines y Umbrales).


# Empaquetado del modelo

In [19]:
artefacto_final = {
    'modelo': modelo_final_xgb,         # Tu modelo final entrenado
    'umbral_optimo': 0.76   # El umbral calibrado para evitar falsas alarmas
}

joblib.dump(artefacto_final, "../modelos/modelo_empaquetado.pkl")

['../modelos/modelo_empaquetado.pkl']